# Phase 3: Inference and Evaluation

This notebook demonstrates how to load the trained LISSclear model, run inference on the held-out test set, calculate quantitative metrics (SSIM, PSNR, SAM, NDVI-MAE), and generate qualitative before/after visualisations.

In [ ]:
!pip install -r ../requirements.txt -q
import sys; sys.path.append('..')
import torch
import numpy as np
from IPython.display import Image, display

from src.utils.config import default_config
from src.models.inpainting_model import LISSClearModel
from src.evaluation.evaluator import Evaluator
from src.evaluation.visualizer import ResultVisualizer
from src.utils.image_utils import ImageUtils

## 1. Load Trained Model

In [ ]:
cfg = default_config
device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Loading model to {device}...")
model = LISSClearModel(
    n_refs=cfg.n_reference_frames,
    freeze_encoder=True
)

model.load_pretrained(cfg.sd2_model_id)
best_model_path = cfg.inpainting_ckpt / "best_model.pt"

if best_model_path.exists():
    print(f"Loading fine-tuned weights from {best_model_path}")
    model.load_checkpoint(best_model_path)
else:
    print("WARNING: Fine-tuned checkpoint not found. Using pretrained SD-2 base (demo mode).")

model = model.to(device)
model.eval();

## 2. Quantitative Evaluation on Test Set

In [ ]:
evaluator = Evaluator(
    model=model,
    patches_dir=cfg.processed_patches,
    device=device,
    batch_size=4,
    split="val"
)

# Uncomment to run full evaluation loop
# metrics = evaluator.evaluate(output_dir=cfg.base_dir / "outputs/metrics")
# print(evaluator.metrics.format_report(metrics))

## 3. Qualitative Visualisation (Single Example)

In [ ]:
# Get a single batch from the val set
batch = next(iter(evaluator.loader))
cloudy, mask, refs, target = [t[0] for t in batch]  # Take first item in batch

# Run inference
result = evaluator.evaluate_single(cloudy, mask, refs, target)
pred = result["output"]
batch_metrics = result.get("metrics", {})

# Visualize
vis = ResultVisualizer(output_dir=cfg.base_dir / "outputs/figures")

# 1. Standard RGB Comparison
fig1 = vis.comparison_figure(
    cloudy=ImageUtils.tensor_to_numpy(cloudy),
    cloud_mask=ImageUtils.tensor_to_numpy(mask).squeeze(0),
    predicted=ImageUtils.tensor_to_numpy(pred),
    target=ImageUtils.tensor_to_numpy(target),
    metrics=batch_metrics,
    save_name="demo_reconstruction"
)
display(Image(filename=cfg.base_dir / "outputs/figures/demo_reconstruction.png"))

# 2. NDVI Preservation Analysis
if cloudy.shape[0] >= 4:
    fig2 = vis.ndvi_comparison(
        pred=ImageUtils.tensor_to_numpy(pred),
        target=ImageUtils.tensor_to_numpy(target),
        mask=ImageUtils.tensor_to_numpy(mask).squeeze(0),
        save_name="demo_ndvi_analysis"
    )
    display(Image(filename=cfg.base_dir / "outputs/figures/demo_ndvi_analysis.png"))